In [ ]:
import os, glob, re, string, pickle
import pandas as pd
import numpy as np
from numpy.linalg import norm
from gensim.models import KeyedVectors

# 0. LOAD FASTTEXT EMBEDDINGS ONCE
print('Loading FastText embeddings…')
ft_model = KeyedVectors.load_word2vec_format('/Users/harshit/Desktop/cc.en.300.vec', binary=False)
print('FastText loaded:', len(ft_model.key_to_index), 'words,', ft_model.vector_size, 'dims')


In [ ]:
# 1. PATH CONFIGURATION
base_data   = '/Users/harshit/Desktop/KelloggXParlamint/Kellogg/data'
country     = 'UA'   # Change as per country
corpus_dir  = os.path.join(base_data, 'raw', f'ParlaMint-{country}-en.txt')
results_dir = os.path.join(base_data, 'results', 'meta_transcript', country)
os.makedirs(results_dir, exist_ok=True)


In [ ]:
# 2. LOAD ALL META-TSV FILES
#
#    Confirmed file structure (from real sample):
#      Column 'ID'         = utterance-level ID, e.g. ParlaMint-AT_1996-01-15-..._d7e826
#                            This EXACTLY matches the line prefix in the .txt transcript files.
#                            Note: the meta ID has NO '-en' in it.
#      Column 'Speaker_ID' = e.g. PAD_00334
#
#    We build:
#      utterance_to_meta : { utterance_ID -> {Speaker_ID, _year, gender, ...} }
#      speaker_meta      : { Speaker_ID   -> {gender, party_status, ...} }  (deduped)

meta_frames = []
for root, _, files in os.walk(corpus_dir):
    for fname in files:
        if fname.endswith('-meta.tsv'):
            try:
                mdf = pd.read_csv(os.path.join(root, fname), sep='\t', index_col=False)
                yr_match = re.search(r'_(\d{4})-\d{2}-\d{2}', fname)
                mdf['_year'] = yr_match.group(1) if yr_match else ''
                meta_frames.append(mdf)
            except Exception as e:
                print(f'  Warning: skipping {fname} ({e})')

meta_all = pd.concat(meta_frames, ignore_index=True)

for col in ['Speaker_gender', 'Party_status', 'Party_orientation', 'Speaker_birth']:
    if col in meta_all.columns:
        meta_all[col] = meta_all[col].fillna('').astype(str)

# utterance_ID -> speaker info + year
utterance_to_meta = (
    meta_all
    .set_index('ID')[['Speaker_ID', '_year', 'Speaker_gender',
                       'Party_status', 'Party_orientation', 'Speaker_birth']]
    .to_dict('index')
)

# Speaker-level lookup, deduplicated
speaker_meta = (
    meta_all
    .drop_duplicates(subset='Speaker_ID')
    .set_index('Speaker_ID')[['Speaker_gender', 'Party_status', 'Party_orientation', 'Speaker_birth']]
    .to_dict('index')
)

print(f'Meta loaded: {len(utterance_to_meta):,} utterance rows | {len(speaker_meta):,} unique speakers')


In [ ]:
# 3. DEFINE DIMENSION GROUPINGS
#    build_speaker_groups(dimension) -> { group_label : set_of_Speaker_IDs }

gen_ranges = {
    'Silent_Generation': (1928, 1945),
    'Baby_Boomers':      (1946, 1964),
    'Generation_X':      (1965, 1980),
    'Millennials':       (1981, 1996),
    'Generation_Z':      (1997, 2012),
}

def get_generation(birth_str):
    try:
        year = int(float(birth_str))
    except:
        return None
    for label, (s, e) in gen_ranges.items():
        if s <= year <= e:
            return label
    return None

def build_speaker_groups(dimension):
    groups = {}
    for spk_id, row in speaker_meta.items():
        if dimension == 'gender':
            g = row.get('Speaker_gender', '').strip().upper()
            label = 'Male' if g == 'M' else ('Female' if g == 'F' else None)
        elif dimension == 'alignment':
            s = row.get('Party_status', '').strip()
            label = s if s in ('Coalition', 'Opposition') else None
        elif dimension == 'ideology':
            o = row.get('Party_orientation', '').strip().lower()
            if 'left' in o:              label = 'Left'
            elif 'right' in o:           label = 'Right'
            elif 'centre' in o or 'center' in o: label = 'Centre'
            else:                        label = None
        elif dimension == 'generation':
            label = get_generation(row.get('Speaker_birth', ''))
        else:
            label = None
        if label:
            groups.setdefault(label, set()).add(spk_id)
    return groups

DIMENSIONS = ['gender', 'alignment', 'ideology', 'generation']
print('Dimension groupings defined.')


In [ ]:
# 4. LOAD & PARSE ALL TRANSCRIPT .TXT FILES
#
#    Each .txt file has tab-separated lines:
#      <utterance_ID>\t<speech text>
#
#    Example:
#      ParlaMint-AT_1996-01-15-020-XX-NRSITZ-00001_d7e826\tPlease take a seat...
#
#    The utterance_ID is looked up in utterance_to_meta to get Speaker_ID + year.
#    Lines with no matching meta entry (procedural chair lines etc.) are skipped.

speech_records     = []
skipped_utterances = 0

txt_files = [
    p for p in glob.glob(os.path.join(corpus_dir, '*/*.txt'))
    if not p.endswith('-meta.tsv')
]

for path in txt_files:
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or '\t' not in line:
                continue
            utterance_id, _, text = line.partition('\t')
            utterance_id = utterance_id.strip()

            if utterance_id not in utterance_to_meta:
                skipped_utterances += 1
                continue

            meta_row = utterance_to_meta[utterance_id]
            spk_id   = meta_row['Speaker_ID']
            year     = meta_row['_year']

            clean = text.lower()
            clean = re.sub(f'[{re.escape(string.punctuation)}]', ' ', clean)
            clean = re.sub(r'\s+', ' ', clean).strip()

            speech_records.append({'speaker_id': spk_id, 'year': year, 'clean_text': clean})

df_speeches = pd.DataFrame(speech_records)
print(f'Loaded {len(df_speeches):,} utterances | {df_speeches["speaker_id"].nunique():,} unique speakers')
print(f'Skipped {skipped_utterances:,} utterances with no matching meta entry')


In [ ]:
# 5. DEFINE WORD GROUPS (unchanged from original)
nature_terms     = ['nature','climate','environment','land','forest','forests',
                    'biodiversity','restoration','reforestation','ecology']
importance_terms = ['important','importance','significant','meaningful']
window = 5

# Static importance embeddings — computed once, reused across all groups
imp_embeddings = {
    imp: ft_model[imp]
    for imp in importance_terms
    if imp in ft_model.key_to_index
}
print(f'Importance embeddings ready: {list(imp_embeddings.keys())}')


In [ ]:
# 6. CORE ANALYSIS FUNCTION
#    Accepts any subset of df_speeches and runs the full pipeline.
#    Output folder: results_dir / <dimension> / <group_label> /

def run_nature_importance(subset_df, dimension, group_label):
    out_dir = os.path.join(results_dir, dimension, group_label)
    os.makedirs(out_dir, exist_ok=True)

    years_sorted = sorted(subset_df['year'].unique())
    if not years_sorted:
        print(f'  [SKIP] {dimension}/{group_label} — no utterances found')
        return

    # Extract context windows
    contexts_by_term_year = {
        term: {yr: [] for yr in years_sorted}
        for term in nature_terms
    }
    for _, row in subset_df.iterrows():
        yr     = row['year']
        tokens = row['clean_text'].split()
        for i, tok in enumerate(tokens):
            if tok in nature_terms:
                start = max(0, i - window)
                end   = min(len(tokens), i + window + 1)
                ctx   = tokens[start:i] + tokens[i+1:end]
                contexts_by_term_year[tok][yr].append(ctx)

    with open(os.path.join(out_dir, 'contexts_by_term_year.pkl'), 'wb') as f:
        pickle.dump(contexts_by_term_year, f)

    ctx_rows = [
        {'year': yr, 'term': term, 'context': ' '.join(ctx)}
        for term, yd in contexts_by_term_year.items()
        for yr, ctxs in yd.items()
        for ctx in ctxs
    ]
    pd.DataFrame(ctx_rows).to_csv(os.path.join(out_dir, 'contexts_by_term_year.csv'), index=False)

    # Compute ALC embeddings
    alc_embeddings = {term: {} for term in nature_terms}
    for term, year_dict in contexts_by_term_year.items():
        for yr, ctxs in year_dict.items():
            inst_vecs = []
            for ctx in ctxs:
                vs = [ft_model[w] for w in ctx if w in ft_model.key_to_index]
                if vs:
                    inst_vecs.append(np.mean(vs, axis=0))
            if inst_vecs:
                alc_embeddings[term][yr] = np.mean(inst_vecs, axis=0)

    with open(os.path.join(out_dir, 'alc_embeddings.pkl'), 'wb') as f:
        pickle.dump(alc_embeddings, f)
    with open(os.path.join(out_dir, 'imp_embeddings.pkl'), 'wb') as f:
        pickle.dump(imp_embeddings, f)

    # Cosine similarity table per year
    for yr in years_sorted:
        data = {}
        for term in nature_terms:
            if yr in alc_embeddings[term]:
                vec_n = alc_embeddings[term][yr]
                sims  = {
                    imp: float(np.dot(vec_n, vec_i) / (norm(vec_n) * norm(vec_i)))
                    for imp, vec_i in imp_embeddings.items()
                }
                data[term] = sims
        df_year = pd.DataFrame.from_dict(data, orient='index').reindex(nature_terms)
        df_year.to_csv(os.path.join(out_dir, f'{yr}_nature_vs_importance.csv'), index=True)

    print(f'  checkmark {dimension}/{group_label} — {len(years_sorted)} years | {len(subset_df):,} utterances')


In [ ]:
# 7. RUN ANALYSIS FOR ALL DIMENSIONS & GROUPS

for dimension in DIMENSIONS:
    print(f'\n=== Dimension: {dimension.upper()} ===')
    speaker_groups = build_speaker_groups(dimension)
    print(f'  Groups found: {list(speaker_groups.keys())}')

    for group_label, speaker_ids in speaker_groups.items():
        subset = df_speeches[df_speeches['speaker_id'].isin(speaker_ids)].copy()
        print(f'  {group_label}: {len(subset):,} utterances from {len(speaker_ids):,} speakers')
        run_nature_importance(subset, dimension, group_label)

print('\n=== All dimensions complete! ===')
